In [1]:
import math
import sys
from pathlib import Path

_here = Path().resolve()
for _p in [_here, *_here.parents]:
    _src = _p / "src"
    if (_src / "qudits_on_qubits" / "__init__.py").is_file():
        repo_root = _p
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
else:
    raise ImportError(
        "qudits_on_qubits repo root not found; run the notebook from notebooks/ or the repo root"
    )

from qiskit.quantum_info import Statevector, Operator, partial_trace, SparsePauliOp
from qiskit import qpy, QuantumCircuit
import numpy as np
from qiskit.synthesis import TwoQubitWeylDecomposition
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Session, Batch

from qudits_on_qubits import create_ame_circuit, generate_b_ame
from sympy.functions.combinatorial.numbers import legendre_symbol
from IPython.display import display, Math
from itertools import product
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from igraph import Graph, plot
import matplotlib.pyplot as plt

In [2]:
selected_candidate = "monomial_full__sup013_P102_ph022"
#QuditsOnQubits\artifacts\iqm_runs\selected_best\two_qutrit\stage2_top10_rerun20_20260706\exact\rank01_monomial_full__sup023_P012_ph022
artifact_circuit_dir = repo_root / "artifacts" / "iqm_runs" / "raw" / "quantum_circuits" / "garnet" / "two_qutrit" / selected_candidate
legacy_circuit_dir = repo_root.parent / "QuditsOnQubits" / "basis_direct_encoding_benchmarks" / "quantum_circuits" / "two_qutrit" / selected_candidate

for circuit_dir in (artifact_circuit_dir, legacy_circuit_dir):
    if (circuit_dir / "graph_state_direct_basis.qpy").is_file():
        break
else:
    raise FileNotFoundError(
        "Circuit artifacts not found. Checked:\n"
        f"- {artifact_circuit_dir}\n"
        f"- {legacy_circuit_dir}"
    )

print(f"Loading circuits from: {circuit_dir}")

with (circuit_dir / "graph_state_direct_basis.qpy").open("rb") as f:
    testqc = qpy.load(f)[0]

with (circuit_dir / "graph_state_direct_basis_transpiled.qpy").open("rb") as f:
    qcsuptrans = qpy.load(f)[0]

with (circuit_dir / "F3_W.qpy").open("rb") as f:
    F3sup = qpy.load(f)[0]

Esup = np.load(circuit_dir / "E.npy")

Loading circuits from: C:\Users\szymo\QuditsOnQubits\QuditsOnQubits\artifacts\iqm_runs\raw\quantum_circuits\garnet\two_qutrit\monomial_full__sup013_P102_ph022


In [3]:
from qudits_on_qubits.bell_measurements.sampler_circuits import build_sampler_circuits_from_graph, build_sampler_circuits_for_candidate
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet
from iqm.iqm_client import CircuitCompilationOptions, DDMode, STANDARD_DD_STRATEGY, DDStrategy
from iqm.qiskit_iqm import IQMProvider
from iqm.qiskit_iqm.iqm_naive_move_pass import transpile_to_IQM
from iqm.iqm_client.transpile import ExistingMoveHandlingOptions

from qudits_on_qubits.bell_measurements.sampler_circuits import run_sampler_circuits_to_counts_by_setting
from qudits_on_qubits.bell_measurements.sampler_circuits import decoding_kwargs_from_metadata
from qudits_on_qubits.bell_measurements.postprocessing import compute_bell_value_from_counts

In [4]:
dd_options = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=STANDARD_DD_STRATEGY,
  )

dd_xy4 = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=DDStrategy(
          gate_sequences=[(5, "YXYX", "asap")]
      ),
  )

In [5]:
provider_garnet = IQMProvider("https://resonance.iqm.tech/", quantum_computer="garnet", token="REMOVED_SECRET")
backend_garnet = provider_garnet.get_backend(use_metrics=True)

In [6]:
layout = qcsuptrans.layout.final_index_layout(filter_ancillas=True)
qutrit_qubits = ((layout[0], layout[1]), (layout[2], layout[3]))

sampler_circuits, metadata = build_sampler_circuits_for_candidate(candidate="two_qutrit", state_circuit=qcsuptrans, E=Esup, qutrit_qubits=qutrit_qubits)
isa_sampler_qc = [transpile_to_IQM(qc, backend=backend_garnet, optimization_level=3, seed_transpiler=9) for qc in sampler_circuits]

In [7]:
qutrit_qubits

((0, 4), (3, 1))

In [8]:
isa_sampler_qc[0].depth()

25

In [9]:
counts_by_setting, run_info = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=AerSampler())

In [10]:
bell_value = compute_bell_value_from_counts(counts_by_setting, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(6.021253178345281-7.36216643204557e-15j)

In [11]:
counts_by_setting_garnet, run_info_garnet = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_xy4})

In [12]:
bell_value = compute_bell_value_from_counts(counts_by_setting_garnet, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(5.153345352541411-6.473988012345444e-15j)

Readout mitigation

In [13]:
from provider import get_backend, get_backend_error_profile, generate_random_error_profile, to_static_architecture

In [14]:
backend = get_backend(quantum_computer="garnet")
error_profile = generate_random_error_profile(backend=backend)

Backend connected successfully, using garnet

Noise profile: random-noise-profile

T1 / T2 / readout
qubit   T1 [us]   T2 [us]  readout 0->1  readout 1->0  readout avg error  readout asymmetry
  QB1 55.310975 21.612741      0.035915      0.030638           0.033277          -0.005277
  QB2 30.665535 27.598982      0.054061      0.060338           0.057199           0.006276
  QB3 71.101610 55.019801      0.035362      0.044252           0.039807           0.008891
  QB4 65.068958 38.588546      0.057209      0.047520           0.052364          -0.009689
  QB5 32.390299 29.151269      0.014217      0.011976           0.013097          -0.002241
  QB6 27.270313 23.993778      0.032064      0.041565           0.036815           0.009501
  QB7 66.675135 21.957254      0.018895      0.013033           0.015964          -0.005862
  QB8 31.545845 28.391260      0.016274      0.014296           0.015285          -0.001978
  QB9 28.890437 12.424546      0.052493      0.061026           0.05675

In [15]:
from iqm.qiskit_iqm.fake_backends.iqm_fake_backend import IQMFakeBackend
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet
garnet = IQMFakeGarnet()
garnet_architecture = garnet.architecture

In [16]:
static_garnet_architecture = to_static_architecture(garnet_architecture)

In [17]:
garnet_noisy_backend = IQMFakeBackend(architecture=static_garnet_architecture, error_profile=error_profile)

In [19]:
counts_by_setting_garnet_noise_model, run_info_garnet_noise_model = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=garnet_noisy_backend)

In [20]:
bell_value = compute_bell_value_from_counts(counts_by_setting_garnet_noise_model, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(3.4230208762052516-4.135580766728708e-15j)

In [21]:
import mthree

In [22]:
mapping = mthree.utils.final_measurement_mapping(isa_sampler_qc[0])

In [23]:
mapping

{0: 19, 1: 15, 2: 18, 3: 14}

In [24]:
measured_physical_qubits = sorted(set(mapping.values()))
measured_physical_qubits

[14, 15, 18, 19]

In [25]:
from qiskit.circuit import QuantumCircuit
import numpy as np

def build_readout_calibration_matrices(
    backend,
    physical_qubits,
    shots=10_000
):
    """
    Zwraca listę macierzy kalibracyjnych M3.
    Dla kubitów, których nie kalibrujemy, pozostaje None.
    """
    matrices = [None] * backend.num_qubits

    for q in physical_qubits:
        # Przygotowanie |0> i pomiar kubitu q
        cal_0 = QuantumCircuit(backend.num_qubits, 1)
        cal_0.measure(q, 0)

        # Przygotowanie |1> i pomiar kubitu q
        cal_1 = QuantumCircuit(backend.num_qubits, 1)
        cal_1.x(q)
        cal_1.measure(q, 0)

        counts_0, counts_1 = backend.run(
            [cal_0, cal_1],
            shots=shots
        ).result().get_counts()

        # P(odczytano 1 | przygotowano 0)
        p10 = counts_0.get("1", 0) / shots

        # P(odczytano 0 | przygotowano 1)
        p01 = counts_1.get("0", 0) / shots

        # Kolumny: stan przygotowany |0>, |1>
        # Wiersze: stan odczytany  |0>, |1>
        matrices[q] = np.array(
            [
                [1 - p10, p01],
                [p10, 1 - p01]
            ],
            dtype=np.float32
        )

        print(f"Qubit {q}:")
        print(matrices[q])

    return matrices

In [76]:
calibration_matrix = build_readout_calibration_matrices(backend_garnet, measured_physical_qubits, shots=10000)

Qubit 14:
[[0.9732 0.0126]
 [0.0268 0.9874]]
Qubit 15:
[[0.9881 0.0244]
 [0.0119 0.9756]]
Qubit 18:
[[0.9837 0.0141]
 [0.0163 0.9859]]
Qubit 19:
[[0.9772 0.013 ]
 [0.0228 0.987 ]]


In [77]:
mit = mthree.M3Mitigation()

mit.cals_from_matrices(calibration_matrix)

In [78]:
quasi = []

for res in list(counts_by_setting_garnet.values()):
    quasi.append(mit.apply_correction(res, mapping, return_mitigation_overhead=True))

In [79]:
quasi_miti = {}

for i, setting in zip(quasi, list(counts_by_setting_garnet.keys())):
    quasi_temp = {}
    for key in i.keys():
        quasi_temp[key] = int(i[key]*1024*20)
    quasi_miti[setting] = quasi_temp

In [80]:
bell_value = compute_bell_value_from_counts(quasi_miti, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(5.551343784423118-6.827871601444713e-15j)

In [81]:
6 * math.cos(math.pi / 9)

5.638155724715451